# Benchmark 2 — 기존 npy 평가 (재생성 없음)

**재생성하지 않고** 각 모델의 **기존 합성데이터(`synthetic_data/<model>/{0..4}.npy`)** 를 그대로 읽어
**gen_quality + TailGAN 방식 VaR/ES** 로만 평가한다. (우리 방식 eval_portfolio EW/MR/TF/VT 는 하지 않음.)

- **ours** = `synthetic_data/baseline/` (ablation 이 만든 Close_to_Close/w10 baseline 데이터).
- baseline 모델 = QuantGAN·TailGAN·Diffusion-TS·DisCorrGAN 의 기존 npy.
- 종합표: `result/statistics/benchmark2_summary_gen_quality.csv`, `benchmark2_summary_tailgan_vares.csv`.

In [1]:
import os, sys, glob
import numpy as np
import pandas as pd
import torch
from utils.utils import *
from utils.metric import *
from utils.file_manager import LOCAL_BASE_DIRECTORY, quantDiffusionPath

file_manager = quantDiffusionPath()
SEQ_LEN, N_RUNS = 256, 5
SD = os.path.join(LOCAL_BASE_DIRECTORY, "result", "statistics", "synthetic_data")

# 평가 대상: label -> 기존 npy 폴더 (재생성 없음)
MODELS = {
    "QuantDiffusion (ours, C2C/w10)": os.path.join(SD, "baseline"),
    "QuantGAN": os.path.join(SD, "quantgan"),
    "TailGAN": os.path.join(SD, "tailgan"),
    "Diffusion-TS": os.path.join(SD, "diffusionTS"),
    "DisCorrGAN": os.path.join(SD, "discorrgan"),
}

# real 기준 = data/indices.csv test 분할 6d (모든 모델 공통)
df = pd.read_csv("./data/indices.csv")
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
df.set_index("Date", inplace=True)
df = df.apply(pd.to_numeric).astype(float)
df_log_returns = np.log1p(df.pct_change()).dropna()
train_size = int(0.7 * len(df_log_returns))
test_raw = df_log_returns.iloc[train_size:, :]
real_data = torch.from_numpy(rolling_window(test_raw.values, SEQ_LEN)).float().transpose(1, 2).cpu().numpy()
print("real_data(test 6d):", real_data.shape)

# TailGAN 방식 평가기
_TG_DIR = os.path.join(LOCAL_BASE_DIRECTORY, "modules", "TailGAN")
if _TG_DIR not in sys.path:
    sys.path.insert(0, _TG_DIR)
import tailgan_eval
_TG = tailgan_eval.configure_from_project(LOCAL_BASE_DIRECTORY)
TG_ALPHAS = (0.05, 0.025, 0.01)


def _pm(m, s, nd=4):
    return f"{m:.{nd}f} ± {s:.{nd}f}" if (np.isfinite(m) and np.isfinite(s)) else ""


def eval_gen_quality(npy_dir, n_runs=N_RUNS):
    windows = [1, 5, 20, 100]
    total = pd.DataFrame()
    for seed in range(n_runs):
        f = os.path.join(npy_dir, f"{seed}.npy")
        if not os.path.exists(f):
            continue
        fake = np.load(f)
        rdf = pd.DataFrame()
        rdf = pd.concat([rdf, calculate_ccf(real_data, fake, 0)], axis=0)
        ccf = pd.DataFrame(columns=["Average"])
        for i in range(1, 11):
            ccf = pd.concat([ccf, calculate_ccf(real_data, fake, i)], axis=1)
        rdf = pd.concat([rdf, pd.DataFrame(ccf.mean(axis=1), columns=["Average"])], axis=0)
        rdf = pd.concat([rdf, calculate_distribution_scores(real_data, fake, windows)], axis=0)
        rdf = pd.concat([rdf, calculate_skew_kurtosis(real_data, fake, windows)], axis=0)
        _, acf_df = calculate_acf_score(real_data, fake, 40)
        rdf = pd.concat([rdf, acf_df], axis=0)
        _, lev_df = calculate_leverage_effect(real_data, fake, 40)
        rdf = pd.concat([rdf, lev_df], axis=0)
        total = pd.concat([total, rdf], axis=1)
    stat = pd.DataFrame(index=total.index, columns=["MEAN", "STD"])
    stat["MEAN"] = total.mean(axis=1).values
    stat["STD"] = total.std(axis=1).values
    return stat


def eval_tailgan(npy_dir, n_runs=N_RUNS):
    per_seed = []
    for seed in range(n_runs):
        f = os.path.join(npy_dir, f"{seed}.npy")
        if not os.path.exists(f):
            continue
        fake = np.load(f).astype(np.float32)
        per_seed.append(tailgan_eval.vares_relerror(_TG.Compute_PNL, _TG.Tensor, real_data, fake, TG_ALPHAS))
    keys = list(per_seed[0].keys())
    arr = np.array([[d[k] for k in keys] for d in per_seed])
    return keys, arr.mean(0), arr.std(0)

c:\Users\Juhwankim\miniconda3\envs\research\lib\site-packages\pandas\core\internals\blocks.py:395: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


real_data(test 6d): (1538, 6, 256)
Namespace(n_epochs=3000, batch_size=1000, lr_D=1e-07, lr_G=1e-06, temp=0.01, b1=0.5, b2=0.999, latent_dim=1000, len=50000, n_rows=5, n_cols=100, n_critic_G=1, n_critic_D=1, static_way='LShort', strategies=['Port', 'MR', 'TF'], n_trans=50, Cap=10, WH=10, ratios=[1.0, 1.0], thresholds_pct=[[31, 69]], data_name='1_Gauss+1_AR50+1_AR-12+1_GARCH-T5+1_GARCH-T10', tickers=['Gauss', 'AR50', 'AR-12', 'GARCH-T5', 'GARCH-T10'], noise_name='t5', alphas=[0.05], W=10.0, score='quant', numNN=10, project=True, version='Test1')


# 평가 + 종합표

In [2]:
# ===== 기존 npy 평가 → 종합표 =====
gq_cols, tg_cols = {}, {}
for label, d in MODELS.items():
    n = len(glob.glob(os.path.join(d, "[0-9]*.npy")))
    if n == 0:
        print(f"  (skip {label}: npy 없음 {d})"); continue
    print(f"평가: {label}  ({n} npy)")
    stat = eval_gen_quality(d)
    gq_cols[label] = pd.Series([_pm(m, s) for m, s in zip(stat["MEAN"], stat["STD"])], index=stat.index)
    keys, m, s = eval_tailgan(d)
    tg_cols[label] = pd.Series([_pm(a, b) for a, b in zip(m, s)], index=keys)

gq_summary = pd.DataFrame(gq_cols)
tg_summary = pd.DataFrame(tg_cols)
gq_summary.to_csv(file_manager.get_result_file("benchmark2_summary_gen_quality"), encoding="utf-8-sig")
tg_summary.to_csv(file_manager.get_result_file("benchmark2_summary_tailgan_vares"), encoding="utf-8-sig")
print("\nsaved: benchmark2_summary_gen_quality.csv, benchmark2_summary_tailgan_vares.csv")
print("\n[Benchmark2 - TailGAN 방식 VaR/ES]")
print(tg_summary.to_string())
print("\n[Benchmark2 - Generation Quality]")
print(gq_summary.to_string())

평가: QuantDiffusion (ours, C2C/w10)  (5 npy)
평가: QuantGAN  (5 npy)
평가: TailGAN  (5 npy)
평가: Diffusion-TS  (5 npy)
평가: DisCorrGAN  (5 npy)

saved: benchmark2_summary_gen_quality.csv, benchmark2_summary_tailgan_vares.csv

[Benchmark2 - TailGAN 방식 VaR/ES]
              QuantDiffusion (ours, C2C/w10)         QuantGAN          TailGAN     Diffusion-TS       DisCorrGAN
RE_VaR(0.05)                 0.3514 ± 0.0054  0.5340 ± 0.0112  0.5343 ± 0.0064  0.5684 ± 0.0048  0.6230 ± 0.0183
RE_ES(0.05)                  0.3276 ± 0.0023  0.6346 ± 0.0213  0.5429 ± 0.0091  0.4527 ± 0.0163  0.8740 ± 0.0118
RE_VaR(0.025)                0.3259 ± 0.0026  0.5951 ± 0.0131  0.5278 ± 0.0061  0.4797 ± 0.0198  0.7834 ± 0.0179
RE_ES(0.025)                 0.3277 ± 0.0038  0.7237 ± 0.0363  0.5842 ± 0.0109  0.3995 ± 0.0189  1.0365 ± 0.0191
RE_VaR(0.01)                 0.3351 ± 0.0034  0.6745 ± 0.0158  0.5605 ± 0.0065  0.4110 ± 0.0180  0.9512 ± 0.0099
RE_ES(0.01)                  0.3388 ± 0.0104  0.8619 ± 0.0753  0.6572 